# Training Neural Networks — The Complete Guide

**What you'll learn in this notebook:**
- Creating synthetic datasets for rapid experimentation
- Writing a training loop from scratch — every line explained
- Adding validation to monitor generalization
- Learning rate scheduling with OneCycleLR
- Gradient clipping for training stability
- Mixed precision training with AMP
- Gradient accumulation for effective large batch sizes
- Plotting training curves (loss and accuracy)
- Transfer learning: freeze a backbone, train a head
- Checkpoint saving and resuming training

**Prerequisites:** Notebooks 01-03 (Tensors, Autograd, Neural Networks).  
**Time:** ~60 minutes  
**Hardware:** CPU only — no GPU required!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 1. Create a Synthetic Dataset

We'll create a simple 2D classification problem with 4 classes arranged in clusters. This lets us train quickly without downloading real data.

In [ ]:
def make_classification_data(n_samples=2000, n_features=20, n_classes=4):
    """Create a synthetic classification dataset with clustered classes."""
    torch.manual_seed(42)
    X_list, y_list = [], []
    for c in range(n_classes):
        center = torch.randn(n_features) * 3
        samples = center + torch.randn(n_samples // n_classes, n_features) * 0.8
        X_list.append(samples)
        y_list.append(torch.full((n_samples // n_classes,), c, dtype=torch.long))
    X = torch.cat(X_list)
    y = torch.cat(y_list)
    perm = torch.randperm(len(X))
    return X[perm], y[perm]

X, y = make_classification_data()
print(f"Features: {X.shape}, Labels: {y.shape}")
print(f"Classes: {torch.unique(y).tolist()}")
print(f"Samples per class: {[(y == c).sum().item() for c in range(4)]}")

dataset = TensorDataset(X, y)
train_data, val_data = random_split(dataset, [1600, 400])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128)

print(f"\nTrain: {len(train_data)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_data)} samples, {len(val_loader)} batches")

## 2. Build the Model

A simple MLP with two hidden layers:

In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_features=20, hidden=64, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = Classifier()
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. The Training Loop — From Scratch

Here's the canonical PyTorch training loop, annotated line by line:

In [ ]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

num_epochs = 10
train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()                        # enable dropout, batchnorm training mode
    epoch_loss, correct, total = 0.0, 0, 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()            # 1. clear gradients from previous step
        logits = model(batch_x)          # 2. forward pass
        loss = criterion(logits, batch_y) # 3. compute loss
        loss.backward()                  # 4. backward pass (compute gradients)
        optimizer.step()                 # 5. update parameters

        epoch_loss += loss.item() * batch_x.size(0)
        correct += (logits.argmax(1) == batch_y).sum().item()
        total += batch_x.size(0)

    train_loss = epoch_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # --- Validation phase ---
    model.eval()                         # disable dropout, use running batchnorm stats
    epoch_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():               # no gradient computation needed
        for batch_x, batch_y in val_loader:
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            epoch_loss += loss.item() * batch_x.size(0)
            correct += (logits.argmax(1) == batch_y).sum().item()
            total += batch_x.size(0)

    val_loss = epoch_loss / total
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

## 4. Plot Training Curves

Always visualize training! Look for:
- **Convergence**: loss going down
- **Overfitting**: train loss goes down but val loss goes up
- **Underfitting**: both losses are high and flat

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label="Train", marker="o", markersize=4)
ax1.plot(val_losses, label="Validation", marker="s", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label="Train", marker="o", markersize=4)
ax2.plot(val_accs, label="Validation", marker="s", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/training_curves.png", dpi=100, bbox_inches="tight")
plt.show()
print("Curves saved.")

## 5. Learning Rate Scheduling

The learning rate is the most important hyperparameter. Schedulers adjust it during training for better convergence.

In [ ]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-2, steps_per_epoch=len(train_loader), epochs=20
)
criterion = nn.CrossEntropyLoss()

lrs = []
train_losses_sched = []

model.train()
for epoch in range(20):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        scheduler.step()  # update LR after each batch (not epoch!)

        lrs.append(optimizer.param_groups[0]["lr"])
        epoch_loss += loss.item()

    train_losses_sched.append(epoch_loss / len(train_loader))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(lrs)
ax1.set_xlabel("Step")
ax1.set_ylabel("Learning Rate")
ax1.set_title("OneCycleLR Schedule")
ax1.grid(True, alpha=0.3)

ax2.plot(train_losses_sched)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Training Loss with OneCycleLR")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/onecycle.png", dpi=100, bbox_inches="tight")
plt.show()

## 6. Gradient Clipping

Prevents exploding gradients by capping the gradient norm. Essential for RNNs and transformers.

In [ ]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
MAX_GRAD_NORM = 1.0

batch_x, batch_y = next(iter(train_loader))

optimizer.zero_grad()
loss = criterion(model(batch_x), batch_y)
loss.backward()

grad_norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float("inf"))

optimizer.zero_grad()
loss = criterion(model(batch_x), batch_y)
loss.backward()
grad_norm_after = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

print(f"Gradient norm before clipping: {grad_norm_before:.4f}")
print(f"Gradient norm after clipping:  {min(grad_norm_after.item(), MAX_GRAD_NORM):.4f}")
print(f"Max allowed norm:              {MAX_GRAD_NORM}")
print(f"\nUsage in training loop:")
print(f"  loss.backward()")
print(f"  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print(f"  optimizer.step()")

## 7. Mixed Precision Training (AMP)

Automatic Mixed Precision uses `float16` for most operations (faster, less memory) and `float32` where precision matters. Even on CPU, this demonstrates the pattern.

In [ ]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# GradScaler is for CUDA; on CPU we just show the autocast pattern
# On GPU you'd add: scaler = torch.amp.GradScaler("cuda")

device = "cpu"

model.train()
batch_x, batch_y = next(iter(train_loader))

# Standard training step (for comparison)
optimizer.zero_grad()
logits = model(batch_x)
print(f"Without AMP: logits dtype = {logits.dtype}")

# With autocast
optimizer.zero_grad()
with torch.autocast(device_type=device, dtype=torch.bfloat16):
    logits = model(batch_x)
    loss = criterion(logits, batch_y)
print(f"With AMP:    logits dtype = {logits.dtype}")
print(f"             loss dtype   = {loss.dtype}")

# On GPU, the full pattern would be:
print("""
# --- Full AMP pattern (GPU) ---
# scaler = torch.amp.GradScaler("cuda")
# for batch_x, batch_y in train_loader:
#     optimizer.zero_grad()
#     with torch.autocast(device_type="cuda", dtype=torch.float16):
#         logits = model(batch_x)
#         loss = criterion(logits, batch_y)
#     scaler.scale(loss).backward()
#     scaler.step(optimizer)
#     scaler.update()
""")

## 8. Gradient Accumulation

When your GPU can't fit a large batch, accumulate gradients over multiple mini-batches to simulate a larger effective batch size.

In [ ]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

ACCUMULATION_STEPS = 4  # effective batch = 64 * 4 = 256

model.train()
optimizer.zero_grad()

for i, (batch_x, batch_y) in enumerate(train_loader):
    logits = model(batch_x)
    loss = criterion(logits, batch_y)
    loss = loss / ACCUMULATION_STEPS  # normalize by accumulation steps
    loss.backward()                    # gradients accumulate

    if (i + 1) % ACCUMULATION_STEPS == 0:
        optimizer.step()               # update only every N steps
        optimizer.zero_grad()

        if (i + 1) // ACCUMULATION_STEPS <= 3:
            print(f"Update {(i+1)//ACCUMULATION_STEPS}: accumulated {ACCUMULATION_STEPS} batches, "
                  f"effective batch={ACCUMULATION_STEPS * 64}")

print(f"\nActual batch size: 64")
print(f"Accumulation steps: {ACCUMULATION_STEPS}")
print(f"Effective batch size: {64 * ACCUMULATION_STEPS}")

## 9. Transfer Learning

The most common pattern in deep learning: take a pretrained backbone, freeze its weights, and train a new classification head.

In [ ]:
class PretrainedBackbone(nn.Module):
    """Simulates a pretrained feature extractor."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

class TransferModel(nn.Module):
    def __init__(self, backbone, num_classes=4):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(32, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

backbone = PretrainedBackbone()
model = TransferModel(backbone)

# Step 1: Freeze backbone
for param in model.backbone.parameters():
    param.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,} (head only)")
print(f"Frozen params:    {total - trainable:,} (backbone)")

# Step 2: Train only the head
optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(5):
    total_loss = 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

# Step 3: Fine-tune everything (unfreeze backbone, lower LR)
print("\nUnfreezing backbone for fine-tuning...")
for param in model.backbone.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lower LR for fine-tuning

## 10. Checkpoint Saving and Resuming

Save everything needed to resume training: model, optimizer, scheduler, and epoch.

In [ ]:
import tempfile, os

model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Simulate training for a few epochs
for epoch in range(3):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = nn.CrossEntropyLoss()(model(bx), by)
        loss.backward()
        optimizer.step()
    scheduler.step()

# Save checkpoint
checkpoint_path = os.path.join(tempfile.gettempdir(), "checkpoint.pth")
checkpoint = {
    "epoch": 3,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_loss": 0.5,
}
torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved: {checkpoint_path} ({os.path.getsize(checkpoint_path):,} bytes)")

# Resume from checkpoint
model2 = Classifier()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=5, gamma=0.5)

ckpt = torch.load(checkpoint_path, weights_only=True)
model2.load_state_dict(ckpt["model_state_dict"])
optimizer2.load_state_dict(ckpt["optimizer_state_dict"])
scheduler2.load_state_dict(ckpt["scheduler_state_dict"])
start_epoch = ckpt["epoch"]

print(f"Resumed from epoch {start_epoch}")
print(f"Current LR: {optimizer2.param_groups[0]['lr']:.6f}")
print(f"Best val loss: {ckpt['best_val_loss']}")

## 11. Complete Production-Ready Training Function

Putting it all together — a reusable training function with all best practices:

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, max_grad_norm=1.0):
    """Production-ready training loop with all best practices."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr * 10, steps_per_epoch=len(train_loader), epochs=epochs
    )
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_val_loss = float("inf")

    for epoch in range(epochs):
        # Train
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for bx, by in train_loader:
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(1) == by).sum().item()
            total += bx.size(0)

        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        # Validate
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                logits = model(bx)
                loss = criterion(logits, by)
                total_loss += loss.item() * bx.size(0)
                correct += (logits.argmax(1) == by).sum().item()
                total += bx.size(0)

        val_loss = total_loss / total
        history["val_loss"].append(val_loss)
        history["val_acc"].append(correct / total)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | Train: {history['train_loss'][-1]:.4f} ({history['train_acc'][-1]:.1%}) | "
                  f"Val: {val_loss:.4f} ({history['val_acc'][-1]:.1%}) | LR: {history['lr'][-1]:.2e}")

    return history

model = Classifier()
history = train_model(model, train_loader, val_loader, epochs=20)

## 🧪 Exercise: Add Early Stopping

**Task:** Modify the training function to stop early if validation loss hasn't improved for `patience` epochs.

Hint: Track the best validation loss and a counter. If the counter exceeds patience, break out of the training loop.

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, epochs=100, lr=1e-3, patience=5):
    """Training with early stopping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        # Train
        model.train()
        for bx, by in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                val_loss += criterion(model(bx), by).item()
        val_loss /= len(val_loader)

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d} | Val Loss: {val_loss:.4f} | "
                  f"Best: {best_val_loss:.4f} | Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}! No improvement for {patience} epochs.")
            model.load_state_dict(best_state)
            break

    return model

model = Classifier()
model = train_with_early_stopping(model, train_loader, val_loader, patience=7)
print("Training complete!")

## Key Takeaways

1. **The training loop has 5 core steps:** zero_grad → forward → loss → backward → step
2. **Always set `model.train()` for training** and `model.eval()` + `torch.no_grad()` for validation
3. **OneCycleLR** is a strong default scheduler — warms up then anneals
4. **Gradient clipping** (`clip_grad_norm_`) prevents exploding gradients
5. **Mixed precision** (autocast + GradScaler) gives ~2x speedup on GPU with no accuracy loss
6. **Gradient accumulation** simulates larger batches by delaying optimizer.step()
7. **Transfer learning**: freeze backbone → train head → optionally fine-tune all
8. **Save checkpoints** with model + optimizer + scheduler state for resumability
9. **Early stopping** prevents overfitting by monitoring validation loss
10. **Always plot training curves** — they tell you what's working and what's not

**Next up:** Notebook 05 — Optimizers and Schedulers Visualized